# WBCBench 2026 — Aşama 11A: RETFound Final Full Fine-Tuning — Seed 42

Final RETFound deneyi. Kernel: **Python (retfound)**. Test split kullanılmaz. Max 6 epoch, patience 2, batch 1, accumulation 4, LR 5e-6. Her epoch sonunda resume checkpoint kaydedilir.


In [1]:
from pathlib import Path
import random,time,json,csv,gc,importlib.util
import numpy as np, torch, timm
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from sklearn.metrics import f1_score,balanced_accuracy_score,accuracy_score
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
device=torch.device("cuda"); assert torch.cuda.is_available()
ROOT=Path.cwd(); MF=ROOT/"RETFound"/"models_vit.py"; PRE=ROOT/"wbc_project"/"checkpoints"/"RETFound_cfp_weights.pth"
TR=ROOT/"wbc_project"/"data"/"processed"/"phase2_train"; VA=ROOT/"wbc_project"/"data"/"processed"/"phase2_eval"
OUT=ROOT/"outputs"/"final"/"retfound_fullft"/"seed42"; OUT.mkdir(parents=True,exist_ok=True)
for p in [MF,PRE,TR,VA]: assert p.exists(),p
print("Torch:",torch.__version__,"timm:",timm.__version__,"GPU:",torch.cuda.get_device_name(0))


C:\Users\sarah\.conda\envs\retfound\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.5.1+cu121 timm: 0.9.2 GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
tt=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(.5),transforms.RandomRotation(10),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
vt=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
td=datasets.ImageFolder(TR,transform=tt); vd=datasets.ImageFolder(VA,transform=vt); assert td.classes==vd.classes and len(td.classes)==13
g=torch.Generator(); g.manual_seed(SEED)
tl=DataLoader(td,batch_size=1,shuffle=True,num_workers=0,pin_memory=True,generator=g); vl=DataLoader(vd,batch_size=2,shuffle=False,num_workers=0,pin_memory=True)
print("Train:",len(td),"Val:",len(vd),"TEST kullanılmıyor.")


Train: 49794 Val: 10700 TEST kullanılmıyor.


In [3]:
spec=importlib.util.spec_from_file_location("rv",MF); rv=importlib.util.module_from_spec(spec); spec.loader.exec_module(rv)
model=rv.RETFound_mae(num_classes=13,drop_path_rate=0.0,global_pool=True)
obj=torch.load(PRE,map_location="cpu",weights_only=False); st=obj.get("model",obj.get("state_dict",obj)) if isinstance(obj,dict) else obj
st={k[7:] if k.startswith("module.") else k:v for k,v in st.items()}; ms=model.state_dict()
comp={k:v for k,v in st.items() if k in ms and tuple(v.shape)==tuple(ms[k].shape) and k not in {"head.weight","head.bias"}}
print("Model keys:",len(ms),"Compatible loaded:",len(comp),f"Coverage={100*len(comp)/len(ms):.2f}%")
model.load_state_dict(comp,strict=False); del obj,st,ms,comp; gc.collect()
for p in model.parameters(): p.requires_grad=True
model=model.to(device)
criterion=nn.CrossEntropyLoss(); opt=torch.optim.AdamW(model.parameters(),lr=5e-6,weight_decay=1e-4); scaler=torch.amp.GradScaler("cuda")
ACCUM=4; MAXE=12; PAT=3
BEST=OUT/"retfound_fullft_seed42_best.pt"; LAST=OUT/"retfound_fullft_seed42_last_resume.pt"
config={"model":"RETFound_mae","seed":42,"adaptation":"full_finetune","max_epochs":12,"patience":3,"batch_size":1,"gradient_accumulation":4,"lr":5e-6,"weight_decay":1e-4,"input_size":224,"selection_metric":"validation_macro_f1","test_used":False,"classes":td.classes}
(OUT/"config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")


Model keys: 296 Compatible loaded: 292 Coverage=98.65%


445

In [4]:
history=[]; best=-1.; bad=0; start=1
if LAST.exists():
 r=torch.load(LAST,map_location="cpu",weights_only=False); model.load_state_dict(r["model_state_dict"]); opt.load_state_dict(r["optimizer_state_dict"]); scaler.load_state_dict(r["scaler_state_dict"])
 history=r["history"]; best=r["best_val_macro_f1"]; bad=r["bad_epochs"]; start=r["epoch"]+1
 print("RESUME:",r["epoch"],"tamamlandı ->",start,"ile devam")
else: print("Yeni seed42 final eğitimi.")


Yeni seed42 final eğitimi.


In [5]:
def train_one():
 model.train(); opt.zero_grad(set_to_none=True); total=n=0
 for i,(x,y) in enumerate(tl,1):
  x=x.to(device); y=y.to(device)
  with torch.autocast("cuda",dtype=torch.float16): z=model(x); raw=criterion(z,y); loss=raw/ACCUM
  scaler.scale(loss).backward()
  if i%ACCUM==0 or i==len(tl): scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
  total+=raw.item()*y.size(0); n+=y.size(0)
  if i%2000==0: print(f" batch {i}/{len(tl)} loss={total/n:.4f} peakVRAM={torch.cuda.max_memory_allocated()/1024**3:.2f}GB")
 return total/n
@torch.inference_mode()
def evaluate():
 model.eval(); ys=[]; ps=[]; total=n=0
 for x,y in vl:
  x=x.to(device); y=y.to(device)
  with torch.autocast("cuda",dtype=torch.float16): z=model(x); loss=criterion(z,y)
  total+=loss.item()*y.size(0); n+=y.size(0); ys+=y.cpu().tolist(); ps+=z.argmax(1).cpu().tolist()
 return total/n,f1_score(ys,ps,average="macro",zero_division=0),balanced_accuracy_score(ys,ps),accuracy_score(ys,ps)


In [6]:
for epoch in range(start,MAXE+1):
 print("\n"+"="*70); print(f"RETFOUND FINAL FULL FT — SEED 42 — EPOCH {epoch}/{MAXE}"); print("="*70); torch.cuda.reset_peak_memory_stats(); t=time.time()
 trainloss=train_one(); valloss,f1,ba,acc=evaluate(); row={"epoch":epoch,"train_loss":trainloss,"val_loss":valloss,"macro_f1":f1,"balanced_acc":ba,"accuracy":acc,"minutes":(time.time()-t)/60}; history.append(row)
 print(f"train_loss={trainloss:.4f} | val_loss={valloss:.4f} | Macro-F1={f1:.4f} | Balanced Acc={ba:.4f} | Acc={acc:.4f} | {row['minutes']:.1f} dk")
 if f1>best:
  best=f1; bad=0; torch.save({"model_state_dict":model.state_dict(),"classes":td.classes,"seed":42,"epoch":epoch,"val_macro_f1":best,"config":config},BEST); print("✓ Yeni BEST")
 else: bad+=1; print(f"İyileşme yok: {bad}/{PAT}")
 torch.save({"model_state_dict":model.state_dict(),"optimizer_state_dict":opt.state_dict(),"scaler_state_dict":scaler.state_dict(),"epoch":epoch,"best_val_macro_f1":best,"bad_epochs":bad,"history":history,"config":config},LAST)
 (OUT/"history.json").write_text(json.dumps(history,indent=2),encoding="utf-8")
 with open(OUT/"history.csv","w",newline="",encoding="utf-8") as f:
  w=csv.DictWriter(f,fieldnames=history[0].keys()); w.writeheader(); w.writerows(history)
 print("✓ Resume checkpoint kaydedildi.")
 if bad>=PAT: print("✓ Early stopping."); break
print("\nRETFOUND FINAL FULL FT — SEED 42 SONUCU"); print("Best validation Macro-F1:",f"{best:.4f}"); print("Epochs completed:",len(history)); print("Best:",BEST); print("✓ TEST SPLIT KULLANILMADI.")



RETFOUND FINAL FULL FT — SEED 42 — EPOCH 1/6
 batch 2000/49794 loss=1.1709 peakVRAM=5.67GB
 batch 4000/49794 loss=0.9473 peakVRAM=5.67GB
 batch 6000/49794 loss=0.8381 peakVRAM=5.67GB
 batch 8000/49794 loss=0.7487 peakVRAM=5.67GB
 batch 10000/49794 loss=0.6891 peakVRAM=5.67GB
 batch 12000/49794 loss=0.6494 peakVRAM=5.67GB
 batch 14000/49794 loss=0.6198 peakVRAM=5.67GB
 batch 16000/49794 loss=0.5949 peakVRAM=5.67GB
 batch 18000/49794 loss=0.5716 peakVRAM=5.67GB
 batch 20000/49794 loss=0.5516 peakVRAM=5.67GB
 batch 22000/49794 loss=0.5326 peakVRAM=5.67GB
 batch 24000/49794 loss=0.5198 peakVRAM=5.67GB
 batch 26000/49794 loss=0.5061 peakVRAM=5.67GB
 batch 28000/49794 loss=0.4934 peakVRAM=5.67GB
 batch 30000/49794 loss=0.4801 peakVRAM=5.67GB
 batch 32000/49794 loss=0.4717 peakVRAM=5.67GB
 batch 34000/49794 loss=0.4636 peakVRAM=5.67GB
 batch 36000/49794 loss=0.4543 peakVRAM=5.67GB
 batch 38000/49794 loss=0.4466 peakVRAM=5.67GB
 batch 40000/49794 loss=0.4402 peakVRAM=5.67GB
 batch 42000/49794